# Task 4: Email Spam Detection
Objective: Build a NLP binary classifier that distinguishes spam emails from legitimate (ham) emails.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import zipfile
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import nltk
from nltk.corpus import stopwords
import string

import warnings
warnings.filterwarnings("ignore")

### Download and Load Dataset
Downloading the SMS Spam Collection dataset from the UCI repository.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = "smsspamcollection.zip"
urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("sms_data")

df = pd.read_csv("sms_data/SMSSpamCollection", sep="\t", header=None, names=["label", "message"])
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nClass Distribution:")
print(df["label"].value_counts())
print("\nPercentage:")
print(df["label"].value_counts(normalize=True) * 100)

### Text Preprocessing
Downloading NLTK stopwords and defining a preprocessing pipeline.

In [ ]:
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = "".join([char for char in text if char not in string.punctuation])
    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_message"] = df["message"].apply(preprocess_text)
df.head()

### Feature Extraction using TF-IDF Vectorizer
**TF-IDF** stands for Term Frequency-Inverse Document Frequency. 
- **Term Frequency (TF)** measures how frequently a term occurs in a document.
- **Inverse Document Frequency (IDF)** measures how important a term is by weighing down frequent terms and scaling up rare ones.

Together, it evaluates how relevant a word is to a document in a collection of documents.

In [ ]:
X = df["clean_message"]
y = df["label"].map({"ham": 0, "spam": 1})

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Train Models
Training a Multinomial Naive Bayes (industry standard for basic text classification) and an SVM.

In [ ]:
nb_clf = MultinomialNB()
nb_clf.fit(X_train_tfidf, y_train)
nb_pred = nb_clf.predict(X_test_tfidf)

svm_clf = SVC(kernel="linear")
svm_clf.fit(X_train_tfidf, y_train)
svm_pred = svm_clf.predict(X_test_tfidf)

### Evaluation

In [ ]:
def evaluate_clf(name, y_true, y_pred):
    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\n")

evaluate_clf("Multinomial Naive Bayes", y_test, nb_pred)
evaluate_clf("Support Vector Machine (Linear)", y_test, svm_pred)

### Why is Recall particularly important for spam detection?
In the context of spam detection, there is a trade-off between Precision and Recall. While high precision ensures that we do not mistakenly classify legitimate emails (ham) as spam, **Recall** is also crucial if the goal is to aggressively catch as much spam as possible. A low recall means a lot of spam makes it into the user's inbox, diminishing the effectiveness of the filter. However, in practice, Precision is often prioritized slightly higher to prevent missing important legitimate emails (false positives).

### (Bonus) WordCloud Visualization

In [ ]:
# pip install wordcloud (uncomment if not installed)
from wordcloud import WordCloud

spam_words = " ".join(df[df["label"] == "spam"]["clean_message"])
ham_words = " ".join(df[df["label"] == "ham"]["clean_message"])

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)
wc_spam = WordCloud(width=400, height=300, background_color="black").generate(spam_words)
plt.imshow(wc_spam, interpolation="bilinear")
plt.title("Spam Words")
plt.axis("off")

plt.subplot(1, 2, 2)
wc_ham = WordCloud(width=400, height=300, background_color="white").generate(ham_words)
plt.imshow(wc_ham, interpolation="bilinear")
plt.title("Ham Words")
plt.axis("off")

plt.show()